# 🚀 IMC Prosperity 4 — Trading Algorithm
**Author:** Albertt | IIT Kharagpur — M.Tech Safety Engineering & Analytics

## 🧠 Strategy Overview
| Product | Strategy | Logic |
|---------|----------|-------|
| `RAINFOREST_RESIN` | **Market Making** | Fair value = 10,000. Post bids/asks ±2 ticks, collect spread |
| `KELP` | **EMA Z-score Mean Reversion** | Short EMA vs Long EMA → z-score → buy low / sell high |
| `SQUID_INK` | **EMA Z-score Mean Reversion** | Same as Kelp but higher volatility |

**Fear & Greed Filter** (applied to all products):  
- Bid-Ask spread > 0.3% of mid price → **Fear mode** → tighter sizing  
- Tight spread → **Greed mode** → scale up order size 1.5×

---
## Cell 1 — Datamodel Stub
> IMC provides a `datamodel` module on their servers. Since it doesn't exist locally, we recreate the classes here so the notebook runs on your machine.

In [ ]:
import sys
import types
import json
import math
import random
import copy
from typing import Dict, List

# ── Build a fake 'datamodel' module that mirrors IMC's real one ──
dm = types.ModuleType("datamodel")

class OrderDepth:
    """Represents one side of the live order book for a product."""
    def __init__(self):
        self.buy_orders:  Dict[int, int] = {}   # {price: qty}  bids
        self.sell_orders: Dict[int, int] = {}   # {price: qty}  asks (negative qty)

class Order:
    """A single order we want to place."""
    def __init__(self, symbol: str, price: int, quantity: int):
        self.symbol   = symbol
        self.price    = price
        self.quantity = quantity   # positive = BUY, negative = SELL
    def __repr__(self):
        side = "BUY " if self.quantity > 0 else "SELL"
        return f"{side} {abs(self.quantity):3d} {self.symbol:<18} @ {self.price}"

class TradingState:
    """Snapshot of the market at each timestamp."""
    def __init__(self, order_depths, position, traderData="", timestamp=0):
        self.order_depths: Dict[str, OrderDepth] = order_depths
        self.position:     Dict[str, int]         = position
        self.own_trades:   Dict[str, list]        = {}
        self.traderData:   str                    = traderData
        self.timestamp:    int                    = timestamp

dm.OrderDepth   = OrderDepth
dm.Order        = Order
dm.TradingState = TradingState
sys.modules["datamodel"] = dm

print("✅ Datamodel stub loaded — OrderDepth, Order, TradingState are ready.")

---
## Cell 2 — Configuration
> Tune these values after reviewing the **Performance** tab on the IMC website.

In [ ]:
# ─── POSITION LIMITS ────────────────────────────────────────────────────────
# Maximum units we can hold (long or short) per product.
POSITION_LIMITS: Dict[str, int] = {
    "RAINFOREST_RESIN": 50,
    "KELP":             50,
    "SQUID_INK":        50,
}

# ─── MARKET MAKING CONFIG ────────────────────────────────────────────────────
MM_FAIR_VALUE  = {"RAINFOREST_RESIN": 10_000}   # Known stable fair value
MM_BASE_SPREAD = 2                               # Post ±2 ticks from fair value
MM_ORDER_SIZE  = 10                              # Units per order leg

# ─── EMA MEAN REVERSION CONFIG ───────────────────────────────────────────────
EMA_SHORT     = 5      # Fast EMA window  (reacts quickly)
EMA_LONG      = 20     # Slow EMA window  (smoother trend)
Z_ENTRY       = 1.5    # Z-score threshold: >+1.5 = overbought, <-1.5 = oversold
MR_ORDER_SIZE = 8      # Units per mean-reversion trade

# ─── FEAR & GREED CONFIG ─────────────────────────────────────────────────────
FEAR_SPREAD_PCT = 0.003   # Spread > 0.3% of mid price → fear mode (cautious)
GREED_SIZE_MULT = 1.5     # Greed mode → scale order sizes by 1.5x

print("✅ Configuration loaded.")
print(f"   Products tracked : {list(POSITION_LIMITS.keys())}")
print(f"   EMA windows      : short={EMA_SHORT}, long={EMA_LONG}")
print(f"   Z-score entry    : ±{Z_ENTRY}")

---
## Cell 3 — Helper Functions

In [ ]:
def ema_update(prev_ema: float, new_price: float, window: int) -> float:
    """
    Exponential Moving Average update.
    α = 2 / (window + 1)  →  more weight on recent prices when window is small.
    """
    alpha = 2.0 / (window + 1)
    return alpha * new_price + (1 - alpha) * prev_ema


def best_bid_ask(order_depth: OrderDepth):
    """
    Returns (best_bid, best_ask) from an OrderDepth.
    best_bid = highest buy price someone is willing to pay
    best_ask = lowest sell price someone is willing to accept
    """
    best_bid = max(order_depth.buy_orders.keys())  if order_depth.buy_orders  else None
    best_ask = min(order_depth.sell_orders.keys()) if order_depth.sell_orders else None
    return best_bid, best_ask


def clamp_qty(product: str, desired_qty: int, current_pos: int) -> int:
    """
    Ensures we never exceed the position limit.
    e.g. if limit=50, pos=45, desired_buy=10 → returns 5 (can only buy 5 more)
    """
    limit = POSITION_LIMITS.get(product, 20)
    if desired_qty > 0:
        return min(desired_qty,  limit - current_pos)
    else:
        return max(desired_qty, -limit - current_pos)


print("✅ Helper functions defined: ema_update, best_bid_ask, clamp_qty")

---
## Cell 4 — Trader Class  ⭐
> **This is the core algo.** The `Trader` class and its `run()` method is what IMC calls on every market tick.
> When you export `trader.py` (Cell 7), this class + helpers get written to the file you upload.

In [ ]:
from datamodel import OrderDepth, TradingState, Order

class Trader:
    """
    IMC Prosperity Trader.
    IMC calls Trader().run(state) every tick and expects:
        return (orders_dict, conversions, trader_data_string)
    """

    def run(self, state: TradingState) -> tuple:

        # ── 1. Load memory from last tick (persisted as JSON string) ────────
        try:
            mem: dict = json.loads(state.traderData) if state.traderData else {}
        except Exception:
            mem = {}

        price_history: dict = mem.get("price_history", {})
        ema_short:     dict = mem.get("ema_short",     {})
        ema_long:      dict = mem.get("ema_long",      {})

        result: Dict[str, List[Order]] = {}

        # ── 2. Process each product on the exchange ──────────────────────────
        for product, order_depth in state.order_depths.items():

            orders:      List[Order] = []
            current_pos: int         = state.position.get(product, 0)

            if not order_depth.buy_orders or not order_depth.sell_orders:
                continue   # no data yet, skip

            best_bid, best_ask = best_bid_ask(order_depth)
            mid_price = (best_bid + best_ask) / 2.0

            # ── 2a. Update rolling price history (last 30 ticks) ────────────
            hist = price_history.get(product, [])
            hist.append(mid_price)
            if len(hist) > 30:
                hist = hist[-30:]
            price_history[product] = hist

            # ── 2b. Update EMAs ──────────────────────────────────────────────
            if product not in ema_short:
                ema_short[product] = mid_price
                ema_long[product]  = mid_price
            else:
                ema_short[product] = ema_update(ema_short[product], mid_price, EMA_SHORT)
                ema_long[product]  = ema_update(ema_long[product],  mid_price, EMA_LONG)

            es = ema_short[product]
            el = ema_long[product]

            # ── 2c. Fear & Greed proxy via bid-ask spread ratio ──────────────
            #        Wide spread  = fear  → cautious (size_mult = 1.0)
            #        Tight spread = greed → aggressive (size_mult = 1.5)
            spread_ratio = (best_ask - best_bid) / mid_price if mid_price > 0 else 0
            fear_mode    = spread_ratio > FEAR_SPREAD_PCT
            size_mult    = 1.0 if fear_mode else GREED_SIZE_MULT

            # ── 2d. Rolling volatility (std dev of last 10 prices) ──────────
            std_dev = 0.0
            if len(hist) >= 5:
                window  = hist[-10:]
                mean_p  = sum(window) / len(window)
                std_dev = math.sqrt(sum((p - mean_p) ** 2 for p in window) / len(window))

            # ── 2e. Choose strategy ──────────────────────────────────────────

            # ═══ STRATEGY A: Market Making (RAINFOREST_RESIN) ════════════════
            if product in MM_FAIR_VALUE:
                fair_val = MM_FAIR_VALUE[product]
                spread   = MM_BASE_SPREAD + (2 if fear_mode else 0)  # widen in fear

                bid_price = fair_val - spread
                ask_price = fair_val + spread

                # Take any ask that is below our bid price (immediate profit)
                if best_ask <= bid_price:
                    qty = clamp_qty(product, int(MM_ORDER_SIZE * size_mult), current_pos)
                    if qty > 0:
                        orders.append(Order(product, best_ask, qty))

                # Take any bid that is above our ask price (immediate profit)
                if best_bid >= ask_price:
                    qty = clamp_qty(product, -int(MM_ORDER_SIZE * size_mult), current_pos)
                    if qty < 0:
                        orders.append(Order(product, best_bid, qty))

                # Post resting orders to passively collect the spread
                buy_qty  = clamp_qty(product,  MM_ORDER_SIZE, current_pos)
                sell_qty = clamp_qty(product, -MM_ORDER_SIZE, current_pos)
                if buy_qty  > 0: orders.append(Order(product, bid_price,  buy_qty))
                if sell_qty < 0: orders.append(Order(product, ask_price, sell_qty))

            # ═══ STRATEGY B: EMA Z-score Mean Reversion (KELP / SQUID_INK) ══
            else:
                if std_dev > 0 and len(hist) >= EMA_LONG:
                    z_score    = (es - el) / std_dev
                    trade_size = int(MR_ORDER_SIZE * size_mult)

                    if z_score > Z_ENTRY:           # OVERBOUGHT → SELL
                        qty = clamp_qty(product, -trade_size, current_pos)
                        if qty < 0:
                            orders.append(Order(product, best_bid, qty))

                    elif z_score < -Z_ENTRY:        # OVERSOLD → BUY
                        qty = clamp_qty(product, trade_size, current_pos)
                        if qty > 0:
                            orders.append(Order(product, best_ask, qty))

                    else:                           # NEUTRAL → passive market make
                        ema_mid     = (es + el) / 2
                        spread_half = max(1, int(std_dev * 0.5))
                        buy_qty  = clamp_qty(product,  MR_ORDER_SIZE, current_pos)
                        sell_qty = clamp_qty(product, -MR_ORDER_SIZE, current_pos)
                        if buy_qty  > 0: orders.append(Order(product, int(ema_mid - spread_half),  buy_qty))
                        if sell_qty < 0: orders.append(Order(product, int(ema_mid + spread_half), sell_qty))

            if orders:
                result[product] = orders

        # ── 3. Save state for next tick ──────────────────────────────────────
        trader_data = json.dumps({
            "price_history": price_history,
            "ema_short":     ema_short,
            "ema_long":      ema_long,
        })

        conversions = 0  # not needed in tutorial round
        return result, conversions, trader_data


print("✅ Trader class defined and ready.")

---
## Cell 5 — Backtester
> Simulates 150 ticks locally. Tracks PnL, position, and orders placed per tick.

In [ ]:
import pandas as pd

# ── Simulation parameters ────────────────────────────────────────────────────
N_TICKS      = 150
PRODUCTS     = ["RAINFOREST_RESIN", "KELP", "SQUID_INK"]

# Starting prices
base_prices  = {"RAINFOREST_RESIN": 10000, "KELP": 5000, "SQUID_INK": 2000}

# Price movement volatility per tick
volatility   = {"RAINFOREST_RESIN": 3, "KELP": 8, "SQUID_INK": 15}

# ── Initialise simulation state ──────────────────────────────────────────────
trader      = Trader()
trader_data = ""
position    = {p: 0 for p in PRODUCTS}
cash        = 0.0
prices      = copy.deepcopy(base_prices)

# Log containers
log_ticks       = []
log_pnl         = []
log_positions   = {p: [] for p in PRODUCTS}
log_prices      = {p: [] for p in PRODUCTS}
log_fear        = []     # 1 = fear mode, 0 = greed mode
log_orders      = []

print(f"Running {N_TICKS}-tick backtest...\n")
print(f"{'Tick':>5} | {'Resin':>7} | {'Kelp':>6} | {'Squid':>6} | {'Cash':>10} | {'PnL':>10} | Orders")
print("-" * 80)

for tick in range(N_TICKS):

    # ── Simulate price movement (random walk) ────────────────────────────────
    for p in PRODUCTS:
        prices[p] += random.randint(-volatility[p], volatility[p])
        prices[p]  = max(prices[p], 100)   # price floor

    # ── Build fake order book (tight spread around current price) ────────────
    order_depths = {}
    spread_fear  = False
    for p in PRODUCTS:
        od = OrderDepth()
        sp = random.choice([1, 1, 1, 2, 2, 5])   # occasionally wide spread
        od.buy_orders  = {prices[p] - sp:  10, prices[p] - sp*2:  5}
        od.sell_orders = {prices[p] + sp: -10, prices[p] + sp*2: -5}
        order_depths[p] = od
        if sp >= 5:
            spread_fear = True

    # ── Run trader ──────────────────────────────────────────────────────────
    state = TradingState(
        order_depths=order_depths,
        position=copy.deepcopy(position),
        traderData=trader_data,
        timestamp=tick,
    )
    orders_dict, conversions, trader_data = trader.run(state)

    # ── Simulate order fills (simple: assume all filled at order price) ──────
    tick_orders = []
    for prod, orders in orders_dict.items():
        for order in orders:
            # Check position limit before filling
            new_pos = position[prod] + order.quantity
            limit   = POSITION_LIMITS.get(prod, 20)
            if abs(new_pos) <= limit:
                position[prod]  = new_pos
                cash           -= order.price * order.quantity   # pay/receive cash
                tick_orders.append(str(order))

    # ── Mark-to-market PnL ──────────────────────────────────────────────────
    portfolio_value = sum(position[p] * prices[p] for p in PRODUCTS)
    pnl             = cash + portfolio_value

    # ── Log ─────────────────────────────────────────────────────────────────
    log_ticks.append(tick)
    log_pnl.append(pnl)
    log_fear.append(1 if spread_fear else 0)
    log_orders.append("; ".join(tick_orders) if tick_orders else "—")
    for p in PRODUCTS:
        log_positions[p].append(position[p])
        log_prices[p].append(prices[p])

    # ── Print every 10 ticks ─────────────────────────────────────────────────
    if tick % 10 == 0 or tick == N_TICKS - 1:
        orders_str = "; ".join(tick_orders[:2]) if tick_orders else "no trades"
        print(f"{tick:5d} | {prices['RAINFOREST_RESIN']:7d} | {prices['KELP']:6d} | "
              f"{prices['SQUID_INK']:6d} | {cash:10.0f} | {pnl:10.2f} | {orders_str}")

print(f"\n{'─'*80}")
print(f"Final PnL          : {pnl:,.2f} SeaShells")
print(f"Final positions    : {position}")
print(f"Fear-mode ticks    : {sum(log_fear)} / {N_TICKS}")

---
## Cell 6 — PnL & Strategy Charts

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(16, 12))
fig.suptitle("IMC Prosperity — Backtest Results", fontsize=16, fontweight="bold", y=0.98)
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

# ── Plot 1: Cumulative PnL ───────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(log_ticks, log_pnl, color="#00d4ff", linewidth=2, label="PnL (SeaShells)")
ax1.axhline(0, color="white", linewidth=0.5, linestyle="--", alpha=0.5)
ax1.fill_between(log_ticks, log_pnl, 0,
                  where=[p >= 0 for p in log_pnl], alpha=0.2, color="green", label="Profit")
ax1.fill_between(log_ticks, log_pnl, 0,
                  where=[p < 0  for p in log_pnl], alpha=0.2, color="red",   label="Loss")
ax1.set_title("📈 Cumulative PnL", fontsize=11)
ax1.set_xlabel("Tick")
ax1.set_ylabel("SeaShells")
ax1.legend(fontsize=8)
ax1.grid(alpha=0.2)

# ── Plot 2: RAINFOREST_RESIN price ───────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(log_ticks, log_prices["RAINFOREST_RESIN"], color="#ffd700", linewidth=1.5)
ax2.axhline(MM_FAIR_VALUE["RAINFOREST_RESIN"], color="white",
             linestyle="--", alpha=0.6, linewidth=1, label="Fair Value 10,000")
ax2.set_title("🪨 Rainforest Resin Price", fontsize=10)
ax2.set_xlabel("Tick")
ax2.set_ylabel("Price")
ax2.legend(fontsize=7)
ax2.grid(alpha=0.2)

# ── Plot 3: KELP price ───────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
ax3.plot(log_ticks, log_prices["KELP"], color="#00ff88", linewidth=1.5)
ax3.set_title("🌿 Kelp Price (EMA Mean Reversion)", fontsize=10)
ax3.set_xlabel("Tick")
ax3.set_ylabel("Price")
ax3.grid(alpha=0.2)

# ── Plot 4: Positions ────────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[2, 0])
colors = ["#00d4ff", "#00ff88", "#ff6b6b"]
for p, c in zip(PRODUCTS, colors):
    ax4.plot(log_ticks, log_positions[p], label=p, color=c, linewidth=1.2)
ax4.axhline(0,  color="white", linewidth=0.5, linestyle="--", alpha=0.4)
ax4.axhline(50, color="red",   linewidth=0.8, linestyle=":",  alpha=0.6, label="Limit +50")
ax4.axhline(-50,color="red",   linewidth=0.8, linestyle=":",  alpha=0.6, label="Limit -50")
ax4.set_title("📊 Positions Over Time", fontsize=10)
ax4.set_xlabel("Tick")
ax4.set_ylabel("Units Held")
ax4.legend(fontsize=7)
ax4.grid(alpha=0.2)

# ── Plot 5: Fear & Greed mode ────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 1])
ax5.fill_between(log_ticks, log_fear, step="mid", color="#ff6b6b", alpha=0.7, label="Fear Mode")
ax5.fill_between(log_ticks, [1 - f for f in log_fear], step="mid",
                  color="#00ff88", alpha=0.4, label="Greed Mode")
ax5.set_title("😱 Fear & Greed Mode (by tick)", fontsize=10)
ax5.set_xlabel("Tick")
ax5.set_yticks([0, 1])
ax5.set_yticklabels(["Greed", "Fear"])
ax5.legend(fontsize=7)
ax5.grid(alpha=0.2)

plt.savefig("backtest_results.png", dpi=150, bbox_inches="tight",
            facecolor="#0d1117", edgecolor="none")
plt.show()
print("✅ Chart saved as backtest_results.png")

---
## Cell 7 — Export `trader.py` for Upload
> Run this cell to write the final `trader.py` file. **This is the file you upload to IMC.**

In [ ]:
trader_py_content = '''
from datamodel import OrderDepth, TradingState, Order
from typing import Dict, List
import json
import math

POSITION_LIMITS = {"RAINFOREST_RESIN": 50, "KELP": 50, "SQUID_INK": 50}
MM_FAIR_VALUE   = {"RAINFOREST_RESIN": 10_000}
MM_BASE_SPREAD  = 2
MM_ORDER_SIZE   = 10
EMA_SHORT       = 5
EMA_LONG        = 20
Z_ENTRY         = 1.5
MR_ORDER_SIZE   = 8
FEAR_SPREAD_PCT = 0.003
GREED_SIZE_MULT = 1.5

def ema_update(prev_ema, new_price, window):
    alpha = 2.0 / (window + 1)
    return alpha * new_price + (1 - alpha) * prev_ema

def best_bid_ask(order_depth):
    best_bid = max(order_depth.buy_orders.keys())  if order_depth.buy_orders  else None
    best_ask = min(order_depth.sell_orders.keys()) if order_depth.sell_orders else None
    return best_bid, best_ask

def clamp_qty(product, desired_qty, current_pos):
    limit = POSITION_LIMITS.get(product, 20)
    if desired_qty > 0:
        return min(desired_qty,  limit - current_pos)
    else:
        return max(desired_qty, -limit - current_pos)

class Trader:
    def run(self, state: TradingState) -> tuple:
        try:
            mem = json.loads(state.traderData) if state.traderData else {}
        except Exception:
            mem = {}
        price_history = mem.get("price_history", {})
        ema_short     = mem.get("ema_short",     {})
        ema_long      = mem.get("ema_long",      {})
        result: Dict[str, List[Order]] = {}
        for product, order_depth in state.order_depths.items():
            orders      = []
            current_pos = state.position.get(product, 0)
            if not order_depth.buy_orders or not order_depth.sell_orders:
                continue
            best_bid, best_ask = best_bid_ask(order_depth)
            mid_price = (best_bid + best_ask) / 2.0
            hist = price_history.get(product, [])
            hist.append(mid_price)
            if len(hist) > 30: hist = hist[-30:]
            price_history[product] = hist
            if product not in ema_short:
                ema_short[product] = mid_price
                ema_long[product]  = mid_price
            else:
                ema_short[product] = ema_update(ema_short[product], mid_price, EMA_SHORT)
                ema_long[product]  = ema_update(ema_long[product],  mid_price, EMA_LONG)
            es = ema_short[product]
            el = ema_long[product]
            spread_ratio = (best_ask - best_bid) / mid_price if mid_price > 0 else 0
            fear_mode    = spread_ratio > FEAR_SPREAD_PCT
            size_mult    = 1.0 if fear_mode else GREED_SIZE_MULT
            std_dev = 0.0
            if len(hist) >= 5:
                window  = hist[-10:]
                mean_p  = sum(window) / len(window)
                std_dev = math.sqrt(sum((p - mean_p)**2 for p in window) / len(window))
            if product in MM_FAIR_VALUE:
                fair_val  = MM_FAIR_VALUE[product]
                spread    = MM_BASE_SPREAD + (2 if fear_mode else 0)
                bid_price = fair_val - spread
                ask_price = fair_val + spread
                if best_ask <= bid_price:
                    qty = clamp_qty(product, int(MM_ORDER_SIZE * size_mult), current_pos)
                    if qty > 0: orders.append(Order(product, best_ask, qty))
                if best_bid >= ask_price:
                    qty = clamp_qty(product, -int(MM_ORDER_SIZE * size_mult), current_pos)
                    if qty < 0: orders.append(Order(product, best_bid, qty))
                buy_qty  = clamp_qty(product,  MM_ORDER_SIZE, current_pos)
                sell_qty = clamp_qty(product, -MM_ORDER_SIZE, current_pos)
                if buy_qty  > 0: orders.append(Order(product, bid_price,  buy_qty))
                if sell_qty < 0: orders.append(Order(product, ask_price, sell_qty))
            else:
                if std_dev > 0 and len(hist) >= EMA_LONG:
                    z_score    = (es - el) / std_dev
                    trade_size = int(MR_ORDER_SIZE * size_mult)
                    if z_score > Z_ENTRY:
                        qty = clamp_qty(product, -trade_size, current_pos)
                        if qty < 0: orders.append(Order(product, best_bid, qty))
                    elif z_score < -Z_ENTRY:
                        qty = clamp_qty(product, trade_size, current_pos)
                        if qty > 0: orders.append(Order(product, best_ask, qty))
                    else:
                        ema_mid     = (es + el) / 2
                        spread_half = max(1, int(std_dev * 0.5))
                        buy_qty  = clamp_qty(product,  MR_ORDER_SIZE, current_pos)
                        sell_qty = clamp_qty(product, -MR_ORDER_SIZE, current_pos)
                        if buy_qty  > 0: orders.append(Order(product, int(ema_mid - spread_half),  buy_qty))
                        if sell_qty < 0: orders.append(Order(product, int(ema_mid + spread_half), sell_qty))
            if orders:
                result[product] = orders
        trader_data = json.dumps({"price_history": price_history,
                                   "ema_short": ema_short, "ema_long": ema_long})
        return result, 0, trader_data
'''.strip()

with open("trader.py", "w") as f:
    f.write(trader_py_content)

import os
size_kb = os.path.getsize("trader.py") / 1024
print(f"✅ trader.py exported successfully!")
print(f"   File size : {size_kb:.2f} KB  (limit = 100 KB — you're fine ✓)")
print(f"   Location  : {os.path.abspath('trader.py')}")
print()
print("📤 NEXT STEPS:")
print("   1. Go to  prosperity.imc.com/game")
print("   2. Click  Upload & Log  tab")
print("   3. Upload trader.py")
print("   4. Check  Performance  tab for simulated PnL")